# Exploratory Data Analysis - Credit Card Balance Dataset

EDA on `credit_card_balance.csv` from the Home Credit Default Risk competition: monthly snapshots of a client's previous Home Credit **credit cards**. This is a panel table — one row per `(SK_ID_PREV, MONTHS_BALANCE)`, so each previous credit contributes many monthly rows.

**Related Notebooks:**
- `application_eda.ipynb` - Application_train dataset
- `bureau_eda.ipynb` - Bureau dataset
- `previous_application_eda.ipynb` - Previous applications

**Analysis Sections:**
1. Setup and Data Loading
2. Basic Dataset Exploration
3. Missing Data Analysis
4. Numeric Features Analysis
5. Categorical Features Analysis
6. Standard Data Quality Checks
7. Functional / Business-Logic Quality Checks

## 1. Setup and Data Loading

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns

# Add src to path to import custom modules
sys.path.insert(0, "/Users/bharathregula/code/home_credit_risk_deeplearn/src")
from home_credit_risk.eda_utils import (
    analyze_categorical_features,
    analyze_numeric_features,
    check_categorical_placeholders,
    check_data_quality,
    check_duplicates,
    detect_sentinel_values,
    display_basic_info,
    show_missing_patterns,
)

# Configure polars display options
pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(-1)  # show all columns
pl.Config.set_fmt_str_lengths(80)

# Set style for plots
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

In [ ]:
# Load dataset
datasets = Path("/Users/bharathregula/code/datasets/home-credit-default-risk")

# Every check below reuses the whole frame, so we collect it once. A large
# infer_schema_length guards against dtype mis-inference across the 23 columns.
cc_df = pl.read_csv(datasets / "credit_card_balance.csv", infer_schema_length=100000)

print("Dataset loaded successfully!")
print(f"Shape: {cc_df.shape}")

In [ ]:
# Quick preview of the data
cc_df.head()

## 2. Basic Dataset Exploration

In [ ]:
# Display basic dataset information
display_basic_info(cc_df, df_name="Credit_Card_Balance")

In [ ]:
# Summary statistics
cc_df.describe()

## 3. Missing Data Analysis

This table has a **composite** primary key `(SK_ID_PREV, MONTHS_BALANCE)` — a single previous credit spans many months — so full-row duplicates are checked here and the composite-key uniqueness is asserted in the functional section.

In [ ]:
# Analyze missing data patterns
missing_summary = show_missing_patterns(cc_df, threshold=0.0)

In [ ]:
# Check for full-row duplicate records
check_duplicates(cc_df)

## 4. Numeric Features Analysis

In [ ]:
# Analyze numeric features
numeric_df = analyze_numeric_features(cc_df, figsize=(18, 6), show_summary=True)

## 5. Categorical Features Analysis

In [ ]:
# Analyze categorical features
analyze_categorical_features(cc_df)

## 6. Standard Data Quality Checks

In [ ]:
# Generic data quality checks
check_data_quality(cc_df)

## 7. Functional / Business-Logic Quality Checks

Cross-field consistency rules specific to a monthly credit-card panel: decomposition identities (a total equals the sum of its parts), monotone orderings of receivable/DPD fields, and count↔amount consistency.

In [ ]:
# Small helper for consistent count + percentage reporting
n = cc_df.height


def pct(count: int) -> str:
    return f"{count:,} ({count / n * 100:.3f}%)"


print(f"Total monthly credit-card rows: {n:,}")

### A. Grain / key integrity

Each `(SK_ID_PREV, MONTHS_BALANCE)` pair should be unique, and `MONTHS_BALANCE` is relative to the application date so it must be ≤ 0.

In [ ]:
# A1. Composite-key uniqueness
grain_dupes = n - cc_df.select("SK_ID_PREV", "MONTHS_BALANCE").n_unique()
print(f"A1. Duplicate (SK_ID_PREV, MONTHS_BALANCE) rows: {pct(grain_dupes)}")

# A2. MONTHS_BALANCE must be <= 0 (past months relative to application)
a2 = cc_df.filter(pl.col("MONTHS_BALANCE") > 0).height
print(f"A2. MONTHS_BALANCE > 0 (future months): {pct(a2)}")

### B. Drawings decomposition identities

The `*_CURRENT` total should equal the sum of its ATM / POS / OTHER parts, for both the amount and the count. Only rows where all parts are non-null are checked.

In [ ]:
# B1. AMT_DRAWINGS_CURRENT == ATM + POS + OTHER
TOL = 0.01
amt_parts = [
    "AMT_DRAWINGS_ATM_CURRENT",
    "AMT_DRAWINGS_POS_CURRENT",
    "AMT_DRAWINGS_OTHER_CURRENT",
]
amt_sub = cc_df.filter(
    pl.all_horizontal(
        [pl.col(c).is_not_null() for c in amt_parts + ["AMT_DRAWINGS_CURRENT"]]
    )
)
b1 = amt_sub.filter(
    (pl.col("AMT_DRAWINGS_CURRENT") - pl.sum_horizontal(amt_parts)).abs() > TOL
)
print(
    f"B1. AMT_DRAWINGS_CURRENT != ATM+POS+OTHER (of {amt_sub.height:,} checked): "
    f"{pct(b1.height)}"
)
b1.select(["SK_ID_PREV", "MONTHS_BALANCE", "AMT_DRAWINGS_CURRENT"] + amt_parts).head(10)

In [ ]:
# B2. CNT_DRAWINGS_CURRENT == ATM + POS + OTHER
cnt_parts = [
    "CNT_DRAWINGS_ATM_CURRENT",
    "CNT_DRAWINGS_POS_CURRENT",
    "CNT_DRAWINGS_OTHER_CURRENT",
]
cnt_sub = cc_df.filter(pl.all_horizontal([pl.col(c).is_not_null() for c in cnt_parts]))
b2 = cnt_sub.filter(
    (pl.col("CNT_DRAWINGS_CURRENT") - pl.sum_horizontal(cnt_parts)).abs() > TOL
)
print(
    f"B2. CNT_DRAWINGS_CURRENT != ATM+POS+OTHER (of {cnt_sub.height:,} checked): "
    f"{pct(b2.height)}"
)

### C. Receivable / balance orderings

Receivable amounts should nest: principal ≤ receivable ≤ total receivable. Balance exceeding the credit limit is reported as *informational* (fees / interest / over-limit usage make it legitimately common).

In [ ]:
# C1. AMT_RECEIVABLE_PRINCIPAL <= AMT_RECIVABLE <= AMT_TOTAL_RECEIVABLE
c1a = cc_df.filter(
    pl.col("AMT_RECEIVABLE_PRINCIPAL") > pl.col("AMT_RECIVABLE") + TOL
).height
c1b = cc_df.filter(
    pl.col("AMT_RECIVABLE") > pl.col("AMT_TOTAL_RECEIVABLE") + TOL
).height
print(f"C1a. PRINCIPAL > RECIVABLE: {pct(c1a)}")
print(f"C1b. RECIVABLE > TOTAL_RECEIVABLE: {pct(c1b)}")

# C2. Balance over the credit limit (informational)
c2 = cc_df.filter(pl.col("AMT_BALANCE") > pl.col("AMT_CREDIT_LIMIT_ACTUAL")).height
print(f"C2. AMT_BALANCE > AMT_CREDIT_LIMIT_ACTUAL (informational): {pct(c2)}")

### D. Days-past-due relationship

`SK_DPD_DEF` is DPD with a tolerance that ignores small debts, so it can only be ≤ the raw `SK_DPD`.

In [ ]:
# D1. SK_DPD_DEF <= SK_DPD
d1 = cc_df.filter(pl.col("SK_DPD_DEF") > pl.col("SK_DPD")).height
print(f"D1. SK_DPD_DEF > SK_DPD: {pct(d1)}")

### E. Non-negativity

Monetary amounts, drawing counts and days-past-due should never be negative (a negative `AMT_BALANCE` means an overpayment / credit balance — worth flagging).

In [ ]:
# E1. Non-negativity across amount / count / DPD columns
nonneg_cols = [
    "AMT_BALANCE",
    "AMT_DRAWINGS_CURRENT",
    "AMT_PAYMENT_CURRENT",
    "AMT_TOTAL_RECEIVABLE",
    "CNT_DRAWINGS_CURRENT",
    "SK_DPD",
    "SK_DPD_DEF",
]
for col in nonneg_cols:
    neg = cc_df.filter(pl.col(col) < 0).height
    if neg:
        print(f"E1. {col} < 0: {pct(neg)}")

### F. Count ↔ amount consistency

For each drawing channel a positive count must come with a positive amount and vice versa (a drawing of zero money, or money with zero drawings, is inconsistent).

In [ ]:
# F1. Count/amount consistency per drawing channel
channels = [
    ("CNT_DRAWINGS_ATM_CURRENT", "AMT_DRAWINGS_ATM_CURRENT"),
    ("CNT_DRAWINGS_POS_CURRENT", "AMT_DRAWINGS_POS_CURRENT"),
    ("CNT_DRAWINGS_OTHER_CURRENT", "AMT_DRAWINGS_OTHER_CURRENT"),
]
for cnt_col, amt_col in channels:
    inconsistent = cc_df.filter(
        ((pl.col(cnt_col) > 0) & (pl.col(amt_col) == 0))
        | ((pl.col(cnt_col) == 0) & (pl.col(amt_col) > 0))
    ).height
    print(f"F1. {cnt_col} vs {amt_col} inconsistent: {pct(inconsistent)}")

### G. Status-driven consistency

A `Completed` credit card should have been paid off, so a non-zero balance on a completed month is suspect.

In [ ]:
# G1. Completed contracts with a non-zero balance
completed = cc_df.filter(pl.col("NAME_CONTRACT_STATUS") == "Completed")
g1 = completed.filter(pl.col("AMT_BALANCE") > 0)
print(
    f"G1. Completed rows with AMT_BALANCE > 0: {g1.height} "
    f"of {completed.height:,} completed"
)
g1.select("SK_ID_PREV", "MONTHS_BALANCE", "AMT_BALANCE", "NAME_CONTRACT_STATUS").head(
    10
)

### H. Sentinel / placeholder detection

Reuse the shared helpers to check for the `365243` numeric sentinel and `XNA`/`XAP` categorical placeholders (this table is expected to be clean on both, which the checks confirm).

In [ ]:
# H1. Numeric sentinel + categorical placeholders.
# detect_sentinel_values auto-skips SK_ID* identifier columns, so an id that
# happens to equal 365243 is not reported as a false positive.
sentinel_summary = detect_sentinel_values(cc_df)
placeholder_summary = check_categorical_placeholders(cc_df)